# Semantic Communication — (PyTorch based)

This notebook sets up the **first building block** for a text-based semantic communication system:
1. Load sentence data (Europarl-style)
2. Build a simple tokenizer + vocabulary
3. Convert text to token IDs
4. Batch with padding using a PyTorch DataLoader
5. Test with your own input sentence

In [27]:
import re
from collections import Counter
from dataclasses import dataclass
from typing import List, Dict

import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cpu


In [28]:
# Local Europarl loader.
from pathlib import Path

USE_LOCAL_EUROPARL = True
LOCAL_EUROPARL_DIR = Path('europarl/en/en')
MAX_SAMPLES = 20000

fallback_sentences = [
    'The european parliament is in session today.',
    'Semantic communication focuses on meaning rather than raw bits.',
    'Neural networks can learn robust representations under noise.',
    'We train an encoder and decoder jointly over a channel model.',
    'The receiver reconstructs text from noisy semantic symbols.',
]

def _read_text_file(path: Path) -> str:
    try:
        return path.read_text(encoding='utf-8')
    except UnicodeDecodeError:
        return path.read_text(encoding='latin-1', errors='ignore')

def _extract_lines_from_text(raw_text: str):
    lines = []
    for line in raw_text.splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith('<') and line.endswith('>'):
            continue
        lines.append(line)
    return lines

def load_sentences_from_local(folder: Path, max_samples: int = 20000):
    txt_files = sorted(folder.glob('*.txt'))
    if not txt_files:
        raise FileNotFoundError(f'No .txt files found in: {folder.resolve()}')

    sentences = []
    for f in txt_files:
        raw = _read_text_file(f)
        lines = _extract_lines_from_text(raw)
        sentences.extend(lines)
        if len(sentences) >= max_samples:
            break

    if len(sentences) == 0:
        raise ValueError('Local Europarl files were found, but no usable sentences were extracted.')

    return sentences[:max_samples]

def load_sentences():
    if USE_LOCAL_EUROPARL:
        try:
            return load_sentences_from_local(LOCAL_EUROPARL_DIR, MAX_SAMPLES)
        except Exception as e:
            print('Local load failed, using fallback sample sentences.')
            print('Reason:', str(e))
            return fallback_sentences

    return fallback_sentences

sentences = load_sentences()
print('Total sentences loaded:', len(sentences))
print('Sample:', sentences[0])
print('Source folder:', LOCAL_EUROPARL_DIR.resolve())

Total sentences loaded: 20000
Sample: Resumption of the session
Source folder: C:\PROJECTS\BTP_Thesis work\europarl\en\en


In [29]:
# Basic tokenizer: lowercase + alphanumeric cleanup + whitespace split
def clean_and_tokenize(text: str) -> List[str]:
    text = text.lower().strip()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    tokens = text.split()
    return tokens

@dataclass
class VocabConfig:
    min_freq: int = 2
    max_vocab_size: int = 20000

class WordVocab:
    def __init__(self, config: VocabConfig):
        self.config = config

        self.PAD = '<PAD>'
        self.UNK = '<UNK>'
        self.BOS = '<BOS>'
        self.EOS = '<EOS>'

        self.word2id: Dict[str, int] = {
            self.PAD: 0,
            self.UNK: 1,
            self.BOS: 2,
            self.EOS: 3,
        }
        self.id2word: Dict[int, str] = {v: k for k, v in self.word2id.items()}

    def build(self, texts: List[str]):
        counter = Counter()
        for t in texts:
            counter.update(clean_and_tokenize(t))

        sorted_words = sorted(counter.items(), key=lambda x: (-x[1], x[0]))

        for word, freq in sorted_words:
            if freq < self.config.min_freq:
                continue
            if len(self.word2id) >= self.config.max_vocab_size:
                break
            idx = len(self.word2id)
            self.word2id[word] = idx
            self.id2word[idx] = word

    def encode(self, text: str, add_bos_eos: bool = True) -> List[int]:
        ids = [self.word2id.get(tok, self.word2id[self.UNK]) for tok in clean_and_tokenize(text)]
        if add_bos_eos:
            ids = [self.word2id[self.BOS]] + ids + [self.word2id[self.EOS]]
        return ids

    def decode(self, ids: List[int], skip_special: bool = True) -> str:
        words = []
        specials = {self.PAD, self.BOS, self.EOS} if skip_special else set()
        for i in ids:
            w = self.id2word.get(int(i), self.UNK)
            if w in specials:
                continue
            words.append(w)
        return ' '.join(words)

vocab = WordVocab(VocabConfig(min_freq=2, max_vocab_size=20000))
vocab.build(sentences)
print('Vocab size:', len(vocab.word2id))

Vocab size: 15869


In [30]:
class EuroparlTextDataset(Dataset):
    def __init__(self, texts: List[str], vocab: WordVocab, max_len: int = 64):
        self.samples = []
        self.vocab = vocab
        self.max_len = max_len

        for t in texts:
            ids = vocab.encode(t, add_bos_eos=True)
            if len(ids) < 3:
                continue

            # Clip while preserving EOS as last token
            if len(ids) > max_len:
                ids = ids[:max_len]
                ids[-1] = vocab.word2id[vocab.EOS]

            # Seq2seq-style teacher forcing targets
            src = torch.tensor(ids[:-1], dtype=torch.long)
            tgt = torch.tensor(ids[1:], dtype=torch.long)
            self.samples.append((src, tgt))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def make_collate_fn(pad_id: int):
    def collate_fn(batch):
        src_list, tgt_list = zip(*batch)
        src_pad = pad_sequence(src_list, batch_first=True, padding_value=pad_id)
        tgt_pad = pad_sequence(tgt_list, batch_first=True, padding_value=pad_id)

        # mask = 1 for real tokens, 0 for pad
        src_mask = (src_pad != pad_id).long()
        return {
            'src_ids': src_pad,
            'tgt_ids': tgt_pad,
            'src_mask': src_mask
        }
    return collate_fn

dataset = EuroparlTextDataset(sentences, vocab=vocab, max_len=64)
loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=make_collate_fn(vocab.word2id[vocab.PAD])
)

print('Dataset size:', len(dataset))
batch = next(iter(loader))
print('src_ids shape:', tuple(batch['src_ids'].shape))
print('tgt_ids shape:', tuple(batch['tgt_ids'].shape))
print('src_mask shape:', tuple(batch['src_mask'].shape))

Dataset size: 19994
src_ids shape: (16, 63)
tgt_ids shape: (16, 63)
src_mask shape: (16, 63)


In [31]:
# Random sanity check: dataset sentence -> token segmentation -> IDs -> decode
import random

sample_sentence = random.choice(sentences)
tokens = clean_and_tokenize(sample_sentence)
encoded = vocab.encode(sample_sentence, add_bos_eos=True)
decoded = vocab.decode(encoded, skip_special=True)

print('Random dataset sentence:')
print(sample_sentence)
print()
print('Token segmentation:')
print(tokens)
print()

token_ids_no_special = encoded[1:-1]
rows = [{'token': tok, 'token_id': tok_id} for tok, tok_id in zip(tokens, token_ids_no_special)]

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    print('Token -> ID mapping (fallback view):')
    for tok, tok_id in zip(tokens, token_ids_no_special):
        print(f'{tok:<20} -> {tok_id}')

print()
print('Full token IDs (with BOS/EOS):', encoded)
print('Decoded text:              ', decoded)

Random dataset sentence:
Mr President, ladies and gentlemen, I think that it has become quite clear today that the situation referred to in the question is of great concern and has attracted the Council' s attention. This is essentially an issue on which there is a degree of agreement, particularly within the European Union. The Portuguese Presidency of the European Union has furthermore taken care to include this issue in its working programme and has ensured that it was a key part of the agenda of the Cairo Euro-African summit. This issue must, however, be considered calmly and carefully, without any histrionics, given the serious situation that the populations of the highly indebted countries find themselves in.

Token segmentation:
['mr', 'president', 'ladies', 'and', 'gentlemen', 'i', 'think', 'that', 'it', 'has', 'become', 'quite', 'clear', 'today', 'that', 'the', 'situation', 'referred', 'to', 'in', 'the', 'question', 'is', 'of', 'great', 'concern', 'and', 'has', 'attracted', 't

,token,token_id
0,mr,29
1,president,44
2,ladies,299
3,and,7
4,gentlemen,295
...,...,...
109,indebted,4434
110,countries,68
111,find,371
112,themselves,523



Full token IDs (with BOS/EOS): [2, 29, 44, 299, 7, 295, 15, 129, 11, 18, 28, 404, 265, 165, 153, 11, 4, 154, 884, 6, 8, 4, 110, 9, 5, 212, 382, 7, 28, 6192, 4, 63, 42, 419, 12, 9, 1936, 33, 122, 16, 19, 41, 9, 10, 1322, 5, 213, 209, 120, 4, 26, 46, 4, 510, 339, 5, 4, 26, 46, 28, 707, 187, 1228, 6, 578, 12, 122, 8, 61, 282, 208, 7, 28, 4551, 11, 18, 52, 10, 560, 183, 5, 4, 402, 5, 4, 2608, 6874, 456, 12, 122, 38, 98, 17, 794, 7550, 7, 1443, 228, 90, 14110, 215, 4, 347, 154, 11, 4, 2845, 5, 4, 1147, 4434, 68, 371, 523, 8, 3]
Decoded text:               mr president ladies and gentlemen i think that it has become quite clear today that the situation referred to in the question is of great concern and has attracted the council s attention this is essentially an issue on which there is a degree of agreement particularly within the european union the portuguese presidency of the european union has furthermore taken care to include this issue in its working programme and has ensured that it 

## Notes for next step
- This notebook is intentionally **word-level and minimal**.
- Next, we add: `SemanticEncoder -> AWGNChannel -> SemanticDecoder` in PyTorch.
- During training, use `CrossEntropyLoss(ignore_index=PAD_ID)` on `tgt_ids`.
- Later, evaluate reconstructed text with BLEU and WER at multiple SNR values.

This cell is a `forward-pass sanity check` for the semantic communication pipeline:

1. src_ids (`token IDs`) are passed into the model.
2. The model creates semantic representations (`encoder`).
3. AWGN noise is added in latent/continuous space (`channel`).
4. The receiver side predicts vocabulary scores (`logits`).
5. We compute cross-entropy loss against target tokens (`tgt_ids`), ignoring <PAD> tokens.

Interpreting the printed shapes
src_ids shape : (16, 63)

* 16 = batch size (16 sentences at once)
* 63 = sequence length (after padding)
logits shape  : (16, 63, 15869)

For every sentence and every token position, the model outputs a score for each vocab word.
15869 = vocabulary size

* Cross-Entropy loss for this batch (lower is better).
Since this is an early/untrained check, a high value is expected.

**Tiny toy example (to visualize what’s happening)**

Assume:

* batch_size = 2
* seq_len = 4
* vocab_size = 6
Then:

src_ids shape = (2, 4)
Example input IDs:

Sentence A: [BOS, the, cat, is]
Sentence B: [BOS, car, is, fast]
logits shape = (2, 4, 6)
Meaning: for each of the 2 × 4 = 8 token positions, model gives 6 scores (one per vocab word).

At one position, logits might look like:
[0.2, 1.1, -0.4, 3.0, 0.6, -1.2]
After softmax, this becomes probabilities over 6 possible words.
Cross-entropy compares that probability distribution with the true next-token ID.

In [32]:
import math
import torch.nn as nn
import torch.nn.functional as F

class AWGNChannel(nn.Module):
    def forward(self, x, snr_db: float):
        # x shape: [B, T, D]
        signal_power = torch.mean(x.pow(2))
        snr_linear = 10 ** (snr_db / 10.0)
        noise_power = signal_power / snr_linear
        noise_std = torch.sqrt(noise_power + 1e-12)
        noise = noise_std * torch.randn_like(x)
        return x + noise

class MiniSemanticComm(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 128, nhead: int = 4, num_layers: int = 2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=4 * d_model,
            dropout=0.1,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.channel = AWGNChannel()
        self.rx_proj = nn.Linear(d_model, d_model)
        self.classifier = nn.Linear(d_model, vocab_size)

    def forward(self, src_ids, src_mask, snr_db: float = 10.0):
        x = self.embed(src_ids)
        # src_key_padding_mask expects True at PAD positions
        key_padding_mask = (src_mask == 0)
        tx_semantic = self.encoder(x, src_key_padding_mask=key_padding_mask)
        rx_semantic = self.channel(tx_semantic, snr_db=snr_db)
        rx_hidden = self.rx_proj(rx_semantic)
        logits = self.classifier(rx_hidden)
        return logits

PAD_ID = vocab.word2id[vocab.PAD]
model = MiniSemanticComm(vocab_size=len(vocab.word2id), d_model=128).to(device)

batch = next(iter(loader))
src_ids = batch['src_ids'].to(device)
tgt_ids = batch['tgt_ids'].to(device)
src_mask = batch['src_mask'].to(device)

logits = model(src_ids, src_mask, snr_db=10.0)
loss = F.cross_entropy(
    logits.reshape(-1, logits.size(-1)),
    tgt_ids.reshape(-1),
    ignore_index=PAD_ID
)

print('src_ids shape :', tuple(src_ids.shape))
print('logits shape  :', tuple(logits.shape))
print('demo CE loss  :', float(loss.detach().cpu()))

src_ids shape : (16, 63)
logits shape  : (16, 63, 15869)
demo CE loss  : 9.716124534606934


till now..

token IDs → encoder → noisy channel → token logits → CE loss

## Train baseline + evaluating across SNR
This section does three things:
1. Split local Europarl sentences into train/validation sets
2. Train `MiniSemanticComm` for a few epochs
3. Evaluate token accuracy, BLEU, and WER at multiple SNR values

In [33]:
# Train/validation split and loaders
import random

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

split_ratio = 0.9
shuffled_sentences = sentences[:]
random.shuffle(shuffled_sentences)
split_idx = int(len(shuffled_sentences) * split_ratio)

train_texts = shuffled_sentences[:split_idx]
val_texts = shuffled_sentences[split_idx:]

train_dataset = EuroparlTextDataset(train_texts, vocab=vocab, max_len=64)
val_dataset = EuroparlTextDataset(val_texts, vocab=vocab, max_len=64)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=make_collate_fn(PAD_ID)
 )

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=make_collate_fn(PAD_ID)
 )

print('Train samples:', len(train_dataset))
print('Val samples  :', len(val_dataset))

Train samples: 17996
Val samples  : 1998


`Train samples`: 17996 = sentences used to learn model weights.

`Val samples`: 1998 = held-out sentences used to check generalization after each epoch.

**follows our 90% training 10% validating scheme**

In [34]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
EPOCHS = 2
TRAIN_SNR_DB = 10.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_train_loss = 0.0
    train_steps = 0

    for batch in train_loader:
        src_ids = batch['src_ids'].to(device)
        tgt_ids = batch['tgt_ids'].to(device)
        src_mask = batch['src_mask'].to(device)

        optimizer.zero_grad()
        logits = model(src_ids, src_mask, snr_db=TRAIN_SNR_DB)
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            tgt_ids.reshape(-1),
            ignore_index=PAD_ID
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_train_loss += float(loss.detach().cpu())
        train_steps += 1

    avg_train_loss = total_train_loss / max(train_steps, 1)

    model.eval()
    total_val_loss = 0.0
    val_steps = 0
    with torch.no_grad():
        for batch in val_loader:
            src_ids = batch['src_ids'].to(device)
            tgt_ids = batch['tgt_ids'].to(device)
            src_mask = batch['src_mask'].to(device)

            logits = model(src_ids, src_mask, snr_db=TRAIN_SNR_DB)
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                tgt_ids.reshape(-1),
                ignore_index=PAD_ID
            )
            total_val_loss += float(loss.detach().cpu())
            val_steps += 1

    avg_val_loss = total_val_loss / max(val_steps, 1)
    print(f'Epoch {epoch}/{EPOCHS} | train_loss={avg_train_loss:.4f} | val_loss={avg_val_loss:.4f}')

Epoch 1/2 | train_loss=5.4514 | val_loss=4.9240
Epoch 2/2 | train_loss=4.7284 | val_loss=4.6002


**Significance of previous step and why I trained on small epochs?**

*Train loss* tells how well model fits seen data.

*Val loss* tells how well it performs on unseen data.

If train loss drops but val loss worsens, that’s overfitting.

## Upgraded training + evaluation

- longer training with early stopping and best-checkpoint restore

In [36]:
import copy

EPOCHS_UPGRADED = 51
PATIENCE = 4
TRAIN_SNR_DB = 10.0
LR = 1e-3

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
best_val_loss = float('inf')
best_state_dict = None
patience_counter = 0

history = []

for epoch in range(1, EPOCHS_UPGRADED + 1):
    model.train()
    total_train_loss = 0.0
    train_steps = 0

    for batch in train_loader:
        src_ids = batch['src_ids'].to(device)
        tgt_ids = batch['tgt_ids'].to(device)
        src_mask = batch['src_mask'].to(device)

        optimizer.zero_grad()
        logits = model(src_ids, src_mask, snr_db=TRAIN_SNR_DB)
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            tgt_ids.reshape(-1),
            ignore_index=PAD_ID
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_train_loss += float(loss.detach().cpu())
        train_steps += 1

    avg_train_loss = total_train_loss / max(train_steps, 1)

    model.eval()
    total_val_loss = 0.0
    val_steps = 0
    with torch.no_grad():
        for batch in val_loader:
            src_ids = batch['src_ids'].to(device)
            tgt_ids = batch['tgt_ids'].to(device)
            src_mask = batch['src_mask'].to(device)

            logits = model(src_ids, src_mask, snr_db=TRAIN_SNR_DB)
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                tgt_ids.reshape(-1),
                ignore_index=PAD_ID
            )
            total_val_loss += float(loss.detach().cpu())
            val_steps += 1

    avg_val_loss = total_val_loss / max(val_steps, 1)
    history.append((epoch, avg_train_loss, avg_val_loss))

    improved = avg_val_loss < best_val_loss
    if improved:
        best_val_loss = avg_val_loss
        best_state_dict = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    print(f'Epoch {epoch:02d}/{EPOCHS_UPGRADED} | train_loss={avg_train_loss:.4f} | val_loss={avg_val_loss:.4f} | best_val={best_val_loss:.4f}')

    if patience_counter >= PATIENCE:
        print(f'Early stopping triggered (patience={PATIENCE}).')
        break

if best_state_dict is not None:
    model.load_state_dict(best_state_dict)
    print('Restored best checkpoint by validation loss.')

Epoch 01/51 | train_loss=4.1471 | val_loss=4.2231 | best_val=4.2231
Epoch 02/51 | train_loss=3.9375 | val_loss=4.1447 | best_val=4.1447


KeyboardInterrupt: 

- Evaluating across SNR values using val loss, token accuracy, BLEU, and WER

In [37]:
import numpy as np
from jiwer import wer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

SNR_LIST = [0.0, 5.0, 10.0, 15.0]
MAX_EVAL_BATCHES = 80  # keep runtime manageable
smoothie = SmoothingFunction().method1

def ids_to_text(ids_tensor):
    # ids_tensor: 1D tensor/list of token ids
    return vocab.decode([int(x) for x in ids_tensor], skip_special=True)

results = []
model.eval()

for snr_db in SNR_LIST:
    total_loss = 0.0
    total_tokens = 0
    correct_tokens = 0
    ref_texts, hyp_texts = [], []

    with torch.no_grad():
        for batch_idx, batch in enumerate(val_loader):
            if batch_idx >= MAX_EVAL_BATCHES:
                break

            src_ids = batch['src_ids'].to(device)
            tgt_ids = batch['tgt_ids'].to(device)
            src_mask = batch['src_mask'].to(device)

            logits = model(src_ids, src_mask, snr_db=snr_db)
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                tgt_ids.reshape(-1),
                ignore_index=PAD_ID
            )
            total_loss += float(loss.detach().cpu())

            pred_ids = torch.argmax(logits, dim=-1)
            non_pad_mask = (tgt_ids != PAD_ID)

            correct_tokens += int(((pred_ids == tgt_ids) & non_pad_mask).sum().item())
            total_tokens += int(non_pad_mask.sum().item())

            # Sentence-level decode for BLEU/WER
            for i in range(tgt_ids.size(0)):
                ref = ids_to_text(tgt_ids[i].detach().cpu().tolist())
                hyp = ids_to_text(pred_ids[i].detach().cpu().tolist())
                if ref.strip():
                    ref_texts.append(ref)
                    hyp_texts.append(hyp)

    avg_val_loss = total_loss / max(min(len(val_loader), MAX_EVAL_BATCHES), 1)
    token_acc = correct_tokens / max(total_tokens, 1)

    bleu_scores = []
    for ref, hyp in zip(ref_texts, hyp_texts):
        ref_tokens = ref.split()
        hyp_tokens = hyp.split()
        if len(hyp_tokens) == 0:
            bleu_scores.append(0.0)
        else:
            bleu_scores.append(
                sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoothie)
            )
    avg_bleu = float(np.mean(bleu_scores)) if bleu_scores else 0.0
    avg_wer = float(wer(ref_texts, hyp_texts)) if ref_texts else 1.0

    results.append({
        'snr_db': snr_db,
        'val_loss': round(avg_val_loss, 4),
        'token_acc': round(token_acc, 4),
        'bleu': round(avg_bleu, 4),
        'wer': round(avg_wer, 4),
        'n_sent': len(ref_texts),
    })

try:
    import pandas as pd
    display(pd.DataFrame(results))
except Exception:
    print(results)

,snr_db,val_loss,token_acc,bleu,wer,n_sent
0,0.0,4.6738,0.2627,0.0577,0.8399,1280
1,5.0,4.2575,0.2982,0.0834,0.7446,1280
2,10.0,4.1314,0.3089,0.1162,0.7072,1280
3,15.0,4.0937,0.3108,0.1299,0.7005,1280


## Final practical demo (easy to understand)
After training, we test on **one real sentence** (same random sentence if available) and inspect:
- Original sentence (transmitter side)
- Reconstructed sentence (receiver side) at different SNR values
- Simple metrics per sentence: BLEU and WER

This is the most intuitive check of how well communication survives noise.

In [38]:
# One-sentence transmitter vs receiver demo (table only)
import random
from jiwer import wer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

model.eval()
smoothie = SmoothingFunction().method1

# Reuse the same earlier sampled sentence if available; otherwise pick one now
demo_sentence = sample_sentence if 'sample_sentence' in globals() else random.choice(sentences)

# Encode exactly like training data (BOS ... EOS), then create src/tgt
demo_ids = vocab.encode(demo_sentence, add_bos_eos=True)
demo_src_ids = torch.tensor([demo_ids[:-1]], dtype=torch.long, device=device)
demo_tgt_ids = torch.tensor([demo_ids[1:]], dtype=torch.long, device=device)
demo_src_mask = (demo_src_ids != PAD_ID).long()

reference_text = vocab.decode(demo_tgt_ids[0].detach().cpu().tolist(), skip_special=True)

snr_demo_list = [0.0, 5.0, 10.0, 15.0]
rows = []

with torch.no_grad():
    for snr_db in snr_demo_list:
        logits = model(demo_src_ids, demo_src_mask, snr_db=snr_db)
        pred_ids = torch.argmax(logits, dim=-1)[0].detach().cpu().tolist()
        predicted_text = vocab.decode(pred_ids, skip_special=True)

        ref_tokens = reference_text.split()
        hyp_tokens = predicted_text.split()
        bleu = sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoothie) if hyp_tokens else 0.0
        one_wer = wer([reference_text], [predicted_text])

        rows.append({
            'snr_db': snr_db,
            'reference_text': reference_text,
            'predicted_text': predicted_text,
            'bleu': float(bleu),
            'wer': float(one_wer),
        })

try:
    import pandas as pd
    pd.set_option('display.max_colwidth', None)
    display(pd.DataFrame(rows))
except Exception:
    print(rows)

,snr_db,reference_text,predicted_text,bleu,wer
0,0.0,mr president ladies and gentlemen i think that it has become quite clear today that the situation referred to in the question is of great concern and has attracted the council s attention this is essentially an issue on which there is a degree of agreement particularly within the european union the portuguese presidency of the european union has furthermore taken care to include this issue in its working programme and has ensured that it was a key part of the agenda of the cairo euroafrican summit this issue must however be considered calmly and carefully without any histrionics given the serious situation that the populations of the highly indebted countries find themselves in,mr president ladies and gentlemen the think that there is considered clear simply that s the european also to include any european is that the concern has gentlemen taken this european has participation within issue something an issue of the has is crucial few of the has in the european union has portuguese presidency has the council union s considered the to of see the issue that this part in that the taken that the is to european part of the portuguese has the european summit i has is of find the considered in in gentlemen is distinction case has this european issue in the situation that the european competent to in it in its,0.124594,0.719298
1,5.0,mr president ladies and gentlemen i think that it has become quite clear today that the situation referred to in the question is of great concern and has attracted the council s attention this is essentially an issue on which there is a degree of agreement particularly within the european union the portuguese presidency of the european union has furthermore taken care to include this issue in its working programme and has ensured that it was a key part of the agenda of the cairo euroafrican summit this issue must however be considered calmly and carefully without any histrionics given the serious situation that the populations of the highly indebted countries find themselves in,mr president ladies and gentlemen the find that there is taken clear clear that that the european in to the this european of a the concern that gentlemen considered this portuguese has attention in issue that a issue of the has is a european of the has in the european union has european presidency has the european union has considered the seriously of the the issue of this agenda in has gentlemen taken that the has considered regular issue of the european and the european summit summit has is of find i considered in the gentlemen has any particular has the european issue in this european that the european constructive situation that a have the,0.114204,0.701754
2,10.0,mr president ladies and gentlemen i think that it has become quite clear today that the situation referred to in the question is of great concern and has attracted the council s attention this is essentially an issue on which there is a degree of agreement particularly within the european union the portuguese presidency of the european union has furthermore taken care to include this issue in its working programme and has ensured that it was a key part of the agenda of the cairo euroafrican summit this issue must however be considered calmly and carefully without any histrionics given the serious situation that the populations of the highly indebted countries find themselves in,mr president ladies and gentlemen i find that the is taken a clear that s the european in to be this european of the the concern that gentlemen taken the european has attention to is a an issue of the has is the european of the with in the european union has european presidency has the european union s considered the to of the the issue of the position in has gentlemen taken that the is a european issue of the european has the european summit and has issue of find the considered in the gentlemen has any particular has the european issue in the

## Next Step (Stronger): explicit encoder-channel-decoder model
Why previous model was limited:
- It learns embeddings locally, but from scratch.
- It predicts tokens position-wise with a weak language prior.
- So long-range context and grammar stay weak.

This next model keeps your same pipeline but improves context modeling:
1. Transformer Encoder builds semantic representation.
2. AWGN channel corrupts that representation.
3. Transformer Decoder generates text autoregressively (uses previous words as context).

In [42]:
# Build a stronger seq2seq semantic communication model
import math
import copy
import torch.nn as nn
import torch.nn.functional as F

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # [1, max_len, d_model]

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class Seq2SeqSemanticComm(nn.Module):
    def __init__(self, vocab_size, pad_id, d_model=256, nhead=8, num_layers=2, dropout=0.1):
        super().__init__()
        self.pad_id = pad_id
        self.d_model = d_model
        self.src_embed = nn.Embedding(vocab_size, d_model)
        self.tgt_embed = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model=d_model, max_len=512)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4*d_model, dropout=dropout, batch_first=True
        )
        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4*d_model, dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_layers)
        self.channel = AWGNChannel()
        self.out = nn.Linear(d_model, vocab_size)

    def _causal_mask(self, tgt_len, device):
        mask = torch.triu(torch.ones(tgt_len, tgt_len, device=device), diagonal=1).bool()
        return mask

    def encode_channel(self, src_ids, src_mask, snr_db):
        src = self.pos(self.src_embed(src_ids) * math.sqrt(self.d_model))
        src_key_padding_mask = (src_mask == 0)
        memory = self.encoder(src, src_key_padding_mask=src_key_padding_mask)
        memory_noisy = self.channel(memory, snr_db=snr_db)
        return memory_noisy, src_key_padding_mask

    def forward(self, src_ids, src_mask, tgt_in_ids, snr_db=10.0):
        memory_noisy, src_key_padding_mask = self.encode_channel(src_ids, src_mask, snr_db)
        tgt = self.pos(self.tgt_embed(tgt_in_ids) * math.sqrt(self.d_model))
        tgt_mask = self._causal_mask(tgt.size(1), tgt.device)
        tgt_key_padding_mask = (tgt_in_ids == self.pad_id)
        dec = self.decoder(
            tgt=tgt,
            memory=memory_noisy,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask
        )
        return self.out(dec)

seq_model = Seq2SeqSemanticComm(
    vocab_size=len(vocab.word2id),
    pad_id=PAD_ID,
    d_model=256,
    nhead=8,
    num_layers=2,
    dropout=0.1
 ).to(device)

print('Seq2Seq model initialized.')

Seq2Seq model initialized.


In [47]:
# Train the stronger seq2seq model (teacher forcing)
EPOCHS_SEQ2SEQ = 8
TRAIN_SNR_DB_SEQ2SEQ = 10.0
optimizer_seq = torch.optim.AdamW(seq_model.parameters(), lr=1e-3)
best_val = float('inf')
best_state = None

for epoch in range(1, EPOCHS_SEQ2SEQ + 1):
    seq_model.train()
    tr_loss = 0.0
    tr_steps = 0
    for batch in train_loader:
        src_ids = batch['src_ids'].to(device)
        tgt_out = batch['tgt_ids'].to(device)
        src_mask = batch['src_mask'].to(device)

        # decoder input uses previous tokens; here src_ids already starts with BOS
        tgt_in = src_ids

        optimizer_seq.zero_grad()
        logits = seq_model(src_ids, src_mask, tgt_in, snr_db=TRAIN_SNR_DB_SEQ2SEQ)
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            tgt_out.reshape(-1),
            ignore_index=PAD_ID
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(seq_model.parameters(), 1.0)
        optimizer_seq.step()

        tr_loss += float(loss.detach().cpu())
        tr_steps += 1

    seq_model.eval()
    va_loss = 0.0
    va_steps = 0
    with torch.no_grad():
        for batch in val_loader:
            src_ids = batch['src_ids'].to(device)
            tgt_out = batch['tgt_ids'].to(device)
            src_mask = batch['src_mask'].to(device)
            tgt_in = src_ids

            logits = seq_model(src_ids, src_mask, tgt_in, snr_db=TRAIN_SNR_DB_SEQ2SEQ)
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                tgt_out.reshape(-1),
                ignore_index=PAD_ID
            )
            va_loss += float(loss.detach().cpu())
            va_steps += 1

    tr_avg = tr_loss / max(tr_steps, 1)
    va_avg = va_loss / max(va_steps, 1)
    if va_avg < best_val:
        best_val = va_avg
        best_state = copy.deepcopy(seq_model.state_dict())

    print(f'Seq2Seq Epoch {epoch:02d}/{EPOCHS_SEQ2SEQ} | train_loss={tr_avg:.4f} | val_loss={va_avg:.4f} | best_val={best_val:.4f}')

if best_state is not None:
    seq_model.load_state_dict(best_state)
    print('Restored best Seq2Seq checkpoint.')

Seq2Seq Epoch 01/8 | train_loss=3.8062 | val_loss=3.9019 | best_val=3.9019
Seq2Seq Epoch 02/8 | train_loss=3.4811 | val_loss=3.7718 | best_val=3.7718
Seq2Seq Epoch 03/8 | train_loss=3.2275 | val_loss=3.6827 | best_val=3.6827
Seq2Seq Epoch 04/8 | train_loss=3.0074 | val_loss=3.5873 | best_val=3.5873
Seq2Seq Epoch 05/8 | train_loss=2.7791 | val_loss=3.4150 | best_val=3.4150
Seq2Seq Epoch 06/8 | train_loss=2.5785 | val_loss=3.3276 | best_val=3.3276
Seq2Seq Epoch 07/8 | train_loss=2.4252 | val_loss=3.2735 | best_val=3.2735


KeyboardInterrupt: 

In [ ]:
# Same-sentence SNR demo with the stronger Seq2Seq model (table only)
from jiwer import wer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

seq_model.eval()
smoothie = SmoothingFunction().method1
BOS_ID = vocab.word2id[vocab.BOS]
EOS_ID = vocab.word2id[vocab.EOS]

demo_sentence = sample_sentence if 'sample_sentence' in globals() else random.choice(sentences)
demo_ids = vocab.encode(demo_sentence, add_bos_eos=True)
src_ids_1 = torch.tensor([demo_ids[:-1]], dtype=torch.long, device=device)
src_mask_1 = (src_ids_1 != PAD_ID).long()
reference_text = vocab.decode(demo_ids[1:], skip_special=True)

def greedy_decode(seq_model, src_ids, src_mask, snr_db, max_len=128):
    memory, mem_pad_mask = seq_model.encode_channel(src_ids, src_mask, snr_db=snr_db)
    generated = [BOS_ID]
    for _ in range(max_len):
        tgt_in = torch.tensor([generated], dtype=torch.long, device=src_ids.device)
        tgt_embed = seq_model.pos(seq_model.tgt_embed(tgt_in) * math.sqrt(seq_model.d_model))
        tgt_mask = seq_model._causal_mask(tgt_in.size(1), src_ids.device)
        dec = seq_model.decoder(
            tgt=tgt_embed,
            memory=memory,
            tgt_mask=tgt_mask,
            memory_key_padding_mask=mem_pad_mask
        )
        logits_step = seq_model.out(dec[:, -1, :])
        next_id = int(torch.argmax(logits_step, dim=-1).item())
        generated.append(next_id)
        if next_id == EOS_ID:
            break
    return generated[1:]  # drop BOS

rows = []
for snr_db in [0.0, 5.0, 10.0, 15.0]:
    pred_ids = greedy_decode(seq_model, src_ids_1, src_mask_1, snr_db=snr_db, max_len=min(len(demo_ids)+20, 160))
    predicted_text = vocab.decode(pred_ids, skip_special=True)

    ref_tokens = reference_text.split()
    hyp_tokens = predicted_text.split()
    bleu = sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoothie) if hyp_tokens else 0.0
    one_wer = wer([reference_text], [predicted_text])

    rows.append({
        'snr_db': snr_db,
        'reference_text': reference_text,
        'predicted_text': predicted_text,
        'bleu': float(bleu),
        'wer': float(one_wer),
    })

try:
    import pandas as pd
    pd.set_option('display.max_colwidth', None)
    display(pd.DataFrame(rows))
except Exception:
    print(rows)